# Risk score

Adds `risk_score` to `regular_season.csv`: for each athlete and season, the share of their races that went wrong.

**risk_score = (DQ + DNF + no-mark + outlier marks) / races started**, counted over the athlete's individual events that season (indoor and outdoor). Every row for that athlete-season gets the same value.

| Counts as a bad race | Codes / rule |
|---|---|
| Disqualified | `DQ`, `FS` (false start) |
| Did not finish | `DNF` |
| No valid field mark | `FOUL`, `NH` (no height), `NM` (no mark): the field-event version of not finishing |
| Outlier mark | A valid mark much worse than the athlete's own season best in that event (see below) |

**Outlier marks.** For every mark, we measure how far off the athlete's season best in that event it was, in %. Normal off-days differ by event: a 100m runner is rarely more than ~3% off, while a javelin thrower is often 10% off. So the cutoff is set per event, from all athletes' marks in that event: **more than 3 interquartile ranges above the 75th percentile** (Tukey's "far out" rule). Examples: about 7% for the 100m, 10% for the 100m hurdles, 32% for the javelin. A mark can only be an outlier if the athlete has at least 2 marks in that event that season.

**Not counted at all:** relays (a DQ or DNF there is a team result), and `DNS`, `NT`, `NP` (the athlete didn't start, or no result was recorded).

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR = Path('../data/processed')
REGULAR_SEASON_PATH = DATA_DIR / 'regular_season.csv'

DQ_CODES = {'DQ', 'FS'}
DNF_CODES = {'DNF'}
NO_MARK_CODES = {'FOUL', 'NH', 'NM'}
NOT_STARTED_CODES = {'DNS', 'NT', 'NP'}
RELAY = r'^(?:4x|DMR|SMR)'
OUTLIER_IQRS = 3  # Tukey's "far out" fence

# Read every column as text so the CSV is written back exactly as it was, plus `risk_score`
regular_season = pd.read_csv(REGULAR_SEASON_PATH, dtype=str, keep_default_na=False)
regular_season = regular_season.drop(columns='risk_score', errors='ignore')


def to_number(mark):
    """'10.74' -> 10.74, '3:44.68' -> 224.68, 'DNF'/'FOUL' -> NaN."""
    try:
        return sum(float(part) * 60**i for i, part in enumerate(reversed(str(mark).strip().split(':'))))
    except ValueError:
        return np.nan


races = regular_season[~regular_season['event'].str.match(RELAY)].copy()
races['mark_code'] = races['mark'].str.strip().str.upper()
races = races[~races['mark_code'].isin(NOT_STARTED_CODES)]
races['mark_num'] = races['mark'].map(to_number)
races['lower_is_better'] = races['event_type'].isin(['track', 'unknown'])  # 'unknown' = odd running distances

# % off the athlete's season best in that event (0 = their best, positive = worse)
valid = races['mark_num'].notna()
group = races[valid].groupby(['athlete_id', 'year', 'event'])['mark_num']
season_best = np.where(races.loc[valid, 'lower_is_better'], group.transform('min'), group.transform('max'))
ratio = races.loc[valid, 'mark_num'] / season_best
races.loc[valid, 'pct_off_best'] = np.where(races.loc[valid, 'lower_is_better'], ratio - 1, 1 - ratio) * 100
races.loc[valid, 'n_marks'] = group.transform('size')

# Per-event outlier cutoff, from marks that aren't the athlete's own best
spread = races[(races['n_marks'] >= 2) & (races['pct_off_best'] > 0)].groupby('event')['pct_off_best']
q1, q3 = spread.quantile(0.25), spread.quantile(0.75)
cutoff = (q3 + OUTLIER_IQRS * (q3 - q1)).rename('outlier_cutoff_pct')
races = races.merge(cutoff, left_on='event', right_index=True, how='left')

races['dq'] = races['mark_code'].isin(DQ_CODES)
races['dnf'] = races['mark_code'].isin(DNF_CODES)
races['no_mark'] = races['mark_code'].isin(NO_MARK_CODES)
races['outlier'] = (races['n_marks'] >= 2) & (races['pct_off_best'] > races['outlier_cutoff_pct'])
races['bad_race'] = races[['dq', 'dnf', 'no_mark', 'outlier']].any(axis=1)

risk = (races.groupby(['athlete_id', 'year'])
        .agg(races=('bad_race', 'size'), dq=('dq', 'sum'), dnf=('dnf', 'sum'),
             no_mark=('no_mark', 'sum'), outlier=('outlier', 'sum'), bad_races=('bad_race', 'sum'))
        .reset_index())
risk['risk_score'] = risk['bad_races'] / risk['races']

print(f'{len(races)} individual races, {races["bad_race"].sum()} bad '
      f'(DQ {races["dq"].sum()}, DNF {races["dnf"].sum()}, no mark {races["no_mark"].sum()}, '
      f'outlier {races["outlier"].sum()})')
print(f'{len(risk)} athlete-seasons; risk_score summary:')
print(risk['risk_score'].describe().round(3).to_string())

35959 individual races, 1411 bad (DQ 51, DNF 340, no mark 874, outlier 146)
3457 athlete-seasons; risk_score summary:
count    3457.000
mean        0.041
std         0.088
min         0.000
25%         0.000
50%         0.000
75%         0.053
max         1.000


In [2]:
out = regular_season.merge(risk[['athlete_id', 'year', 'risk_score']].assign(
                               risk_score=risk['risk_score'].map(lambda r: f'{r:.4f}')),
                           on=['athlete_id', 'year'], how='left', validate='many_to_one')
out['risk_score'] = out['risk_score'].fillna('')  # athlete-seasons with only relays or DNS
assert len(out) == len(regular_season)
out.to_csv(REGULAR_SEASON_PATH, index=False)
print(f'regular_season.csv: risk_score filled for {out["risk_score"].ne("").sum()} of {len(out)} rows')

regular_season.csv: risk_score filled for 41485 of 41493 rows


## Check

In [3]:
print('Outlier cutoffs for the main events (% off season best):')
main = ['100', '200', '400', '800', '1500', '5000', '100H', '110H', '400H', '3000S',
        'HJ', 'PV', 'LJ', 'TJ', 'SP', 'DT', 'HT', 'JT']
display(cutoff.reindex(main).round(1).to_frame().T)

print('Sample of marks flagged as outliers:')
display(races[races['outlier']].sample(12, random_state=0)
        [['athlete_name', 'year', 'event', 'meet_name', 'mark', 'pct_off_best', 'outlier_cutoff_pct']].round(1))

print('Highest risk scores (athlete-seasons with at least 5 races):')
names = races.groupby(['athlete_id', 'year'])['athlete_name'].first()
display(risk[risk['races'] >= 5].join(names, on=['athlete_id', 'year'])
        .sort_values('risk_score', ascending=False).head(10).round(3))

Outlier cutoffs for the main events (% off season best):


event,100,200,400,800,1500,5000,100H,110H,400H,3000S,HJ,PV,LJ,TJ,SP,DT,HT,JT
outlier_cutoff_pct,6.7,7.9,7.7,10.2,8.7,9.4,10.3,8.6,10.7,8.9,13.7,18.8,19.1,13.7,22.9,26.5,22.2,32.1


Sample of marks flagged as outliers:


,athlete_name,year,event,meet_name,mark,pct_off_best,outlier_cutoff_pct
1707,Monea Jennings,2022,SP,Lumberjack Team Challenge,5.52,40.3,22.9
6065,Sydney Barnes,2021,100H,Long Beach Invitational,17.95,20.1,10.3
6332,Drew Pickett,2022,110H,2022 94th Clyde Littlefield Texas Relays,16.61,17.1,8.6
35113,Shaelyn Amos,2022,100H,UCSB Invite Late Afternoon Combined Events,18.67,14.3,10.3
41043,Gracelyn Brown,2025,HT,Beach Invite,36.13,25.0,22.2
14382,Faith Jones,2023,100H,Stanford Invitational,16.13,12.1,10.3
11388,Carter Birade,2023,60H,2023 UW Preview,8.80,8.0,7.7
5956,Ouanessa Nana,2021,800,Cal Poly Spring Break Challenge,2:38.28,10.4,10.2
33547,Zachery Glick,2022,JT,West Coast Relays,23.27,33.4,32.1
6326,Drew Pickett,2022,400H,Beach Invitational 2022,1:05.38,19.9,10.7


Highest risk scores (athlete-seasons with at least 5 races):


,athlete_id,year,races,dq,dnf,no_mark,outlier,bad_races,risk_score,athlete_name
2150,8360079,2026,6,0,0,4,0,4,0.667,Charlotte O'Connor
1027,7753251,2021,7,0,0,4,0,4,0.571,Samuel Heinrichs
2778,8722775,2025,6,0,0,3,0,3,0.500,Madison Pelfrey
2185,8361650,2024,8,0,0,4,0,4,0.500,DeQuan Guillebeau
523,7515270,2021,6,0,0,3,0,3,0.500,Nathan Garibay
1188,7891280,2026,6,0,3,0,0,3,0.500,Jacob Bustamante
109,6615609,2021,6,0,0,3,0,3,0.500,Isaiah Rucker
1108,7767606,2021,6,0,0,3,0,3,0.500,Jada Reid
1642,8020774,2025,9,0,0,4,0,4,0.444,Cyprus Rice
447,7499067,2022,7,0,0,3,0,3,0.429,Jacob Whitby
